# RCS Phase C–G — Kaggle GPU (MARBERTv2-only)

Bundle CSV: `data/gold/gold_train_val_only.csv` — **no human_test rows**.

**GPU setup:** Cell 1 installs a compatible PyTorch (P100/T4). **Restart kernel**, then Cell 2 verifies CUDA + smoke test.

In [ ]:
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input/rcs-training-bundle/rcs_project")
WORK = Path("/kaggle/working/rcs_project")
if INPUT.exists():
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(INPUT, WORK)
else:
    WORK = Path(".").resolve()

os.chdir(WORK)
os.environ["PYTHONPATH"] = "src:."
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
GOLD_CSV = "data/gold/gold_train_val_only.csv"
print("Working directory:", WORK)

In [ ]:
# Cell 1 — Install GPU-compatible PyTorch + ML stack
# REQUIRED: Restart kernel after this cell completes, then run Cell 2.
!python kaggle/setup_gpu_env.py --install

## Restart kernel now

**Session → Restart session** (or Restart kernel), then run the cell below.

In [ ]:
# Cell 2 — After restart: verify CUDA + MARBERTv2 smoke test
import os
from pathlib import Path

WORK = Path("/kaggle/working/rcs_project")
if WORK.exists():
    os.chdir(WORK)
os.environ["PYTHONPATH"] = "src:."
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
GOLD_CSV = "data/gold/gold_train_val_only.csv"

!python kaggle/setup_gpu_env.py --verify
!python scripts/gpu_smoke_test.py

In [ ]:
# Optional: verify splits (no training)
!python scripts/prepare_training_data.py --gold-csv {GOLD_CSV}

In [ ]:
# Full training — run only after smoke test passes
OUTPUT = "/kaggle/working/selection"
!python scripts/run_model_selection.py \
  --gold-csv {GOLD_CSV} \
  --output-dir {OUTPUT} \
  --skip-classical

In [ ]:
!zip -r /kaggle/working/selection.zip /kaggle/working/selection
print("Download selection.zip from Kaggle output panel")